# Trabajo Práctico Integrador

## Port Log - Sprint 1

## Integrantes del grupo 25:


* Oscar Alejandro Franco

## Repositorio GITHUB

Especificar en la siguiente línea cual es la url para acceder al repositorio del grupo.



> https://github.com/oscale05/TRABAJO_PRACTICO_INTEGRADOR_UGR_HS.git



## Consideraciones Generales e Importantes

- Desarrollo base: Los puntos solicitados son la base del trabajo práctico. Es posible la ampliación sin restricciones del mismo.
- Para el traceo se deben agregar 1 [secrets](https://drlee.io/how-to-use-secrets-in-google-colab-for-api-key-protection-a-guide-for-openai-huggingface-and-c1ec9e1277e0) para la utilización de github.
  - GITHUB_TOKEN
- Organización del Notebook:
  - Agregar antes del ejercicio 01 una celda donde se importan todas librerías necesarias.
  - Utilizar un título para cada celda y escribir cada funcionalidad en una celda con título.
- La defensa es individual.
- Los grupos deben mantenerse a lo largo del trabajo integrador.
- La opción `DESACTIVAR_GIT_PUSH` solo será utilizada por los profesores.

## Versionado y Trabajo en Equipo


- Control de Versiones: Utilizar herramientas de versionado
- Colaboradores: Brindar acceso al repositorio a las siguientes cuentas:
  - lcd-had141@ugr.edu.ar
  - fpasinato@ugr.edu.ar
- La descripción del commit debe ser con el siguiente formato `"Día X: \<comentario del commit\>"`
- Changelog: Cada ejercicio/punto del trabajo práctico es considerado como un día distinto de trabajo. Debe quedar constancia en un archivo CHANGELOG.md de los cambios generados cada día. El changelog debe escribirse de tal forma que lo primero que se observe es el último cambio realizado, de esta forma apenas se abre el archivo se pueden leer los últimos cambios realizados. La forma de escritura debe ser un encabezado entre corchetes [Ejercicio X] y luego debajo los cambios realizado como bullets.
- El archivo README.md debe contener el sprint en el que nos encontramos y debe contener:
  - El objetivo.
  - La introducción y el contexto del sprint de trabajo.

## Criterios de evaluación

- Funcionalidad: el código debe ejecutar sin errores al dar "Ejecutar todo" (Ctrl+F9). Validar con Runtime -> Restart session and run all antes de entregar.
- Claridad: código organizado, nombres descriptivos, comentarios solo donde corresponde (funciones, clases, métodos). No se admite `git add .`.
- Estilo: [type hints](https://docs.python.org/3/library/typing.html), [pep8](https://peps.python.org/pep-0008/).

- Herramientas: librerías vistas en la cursada. Sin gdrive ni herramientas alternativas.

## Condiciones de Entrega
- Archivo `.ipynb`, ejecutado y con datos precargados, sobre copia de la plantilla original manteniendo la nomenclatura del nombre.
- Canal: aula virtual. Correo a lcd-had141@ugr.edu.ar solo si el plazo finalizó. No se aceptan enlaces.
- Los trabajos que no cumplan con los requisitos no serán tomados en cuenta como nota final. Solo serán corregidos y evaluados con nota para la devolución correspondiente. Se considera incumplimiento a:
  - Entregas fuera de término.
  - Trabajos que no se entreguen sobre la plantilla de trabajo estipulada.
  - Trabajos que no se ejecuten completos (que contengan errores intermedios al dar "Ejecutar todo").

---



# Objetivo

Aplicar conocimientos de versionado, organización y análisis exploratorio de datos con pandas sobre un dataset real de operaciones portuarias.

## Introducción y Contexto del problema

### Sprint 1

El Puerto Fluvial de Rosario es uno de los complejos portuarios más importantes de América del Sur, siendo el principal punto de exportación de granos y derivados de la Argentina. Diariamente ingresan y egresan decenas de buques de distintas banderas con cargas de diverso tipo.

El sistema de registro de movimientos portuarios fue migrado recientemente desde un sistema heredado de los años '90. Ese sistema acumuló durante décadas inconsistencias de formato en fechas, matrículas de buques y valores numéricos fuera de rango, generando registros que no pueden incorporarse directamente al nuevo sistema.

Nuestro equipo fue contratado para analizar y depurar los datos del sistema antiguo.

Descargar el dataset [port_movements](https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv)  

`https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv`

---

In [ ]:
#@title Agregamos variables de entorno y secrets
import os
import sys

HOOKS = "/tmp/.git-templates/hooks"

# Detectar si estamos en Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    try:
        DESACTIVAR_GIT_PUSH = bool(userdata.get('DESACTIVAR_GIT_PUSH'))
    except userdata.SecretNotFoundError:
        DESACTIVAR_GIT_PUSH = False
else:
    # En local: usar variables de entorno o valores por defecto
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
    DESACTIVAR_GIT_PUSH = os.environ.get('DESACTIVAR_GIT_PUSH', 'False').lower() == 'true'

os.environ['DESACTIVAR_GIT_PUSH'] = str(DESACTIVAR_GIT_PUSH)
os.environ['HOOKS'] = HOOKS
print(f'IN_COLAB: {IN_COLAB}')
print(f'DESACTIVAR_GIT_PUSH: {DESACTIVAR_GIT_PUSH}')
print(f'HOOKS: {HOOKS}')

In [ ]:
#@title Creación del hook para que durante la corrección no se realicen modificaciones al repositorio
import stat

HOOKS = os.environ.get('HOOKS', '/tmp/.git-templates/hooks')
DESACTIVAR_GIT_PUSH = os.environ.get('DESACTIVAR_GIT_PUSH', 'False').lower() == 'true'

os.makedirs(HOOKS, exist_ok=True)
hook_path = os.path.join(HOOKS, 'pre-push')
hook_content = '''#!/bin/sh
if [ "$DESACTIVAR_GIT_PUSH" = "'True'" ] || [ "$DESACTIVAR_GIT_PUSH" = "True" ]; then
    echo "[GIT HOOK - WARNING] Pushes are disabled for this repository."
    exit 1
fi
'''
with open(hook_path, 'w') as f:
    f.write(hook_content)
os.chmod(hook_path, os.stat(hook_path).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
print(f'Hook creado en: {hook_path}')
print(f'DESACTIVAR_GIT_PUSH = {DESACTIVAR_GIT_PUSH}')

In [ ]:
#@title Posicionarse en directorio de trabajo y agregar la siguiente configuración a git
import subprocess

DESACTIVAR_GIT_PUSH = os.environ.get('DESACTIVAR_GIT_PUSH', 'False').lower() == 'true'
HOOKS = os.environ.get('HOOKS', '/tmp/.git-templates/hooks')

if DESACTIVAR_GIT_PUSH:
    result = subprocess.run(['git', 'config', '--global', 'core.hooksPath', HOOKS], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'Hook path configurado: {HOOKS}')
    else:
        print(f'Error: {result.stderr}')
else:
    print('DESACTIVAR_GIT_PUSH es False, no se configura hook')

In [ ]:
#@title Imports y configuración inicial
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime
import os
import sys
from pathlib import Path
import warnings
import json
import requests
from scipy.stats import gaussian_kde
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Verificación de dependencias
print(f"pandas {pd.__version__}")
print(f"numpy {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"seaborn {sns.__version__}")
print(f"requests {requests.__version__}")
print("✅ Todas las dependencias disponibles")

## Ejercicio 01

> Puntos: 1

Inicialización y configuración de la herramienta de versionado sobre la rama `Sprint_1`.

La estructura de directorios es:

```
├── port_log
│   ├── data
│   │   ├── interim    (datasets procesados en pasos intermedios)
│   │   │   ├── plots
│   │   ├── processed  (datasets finales para otra aplicación)
│   │   ├── raw        (datasets en crudo)
│   ├── reports        (resúmenes estadísticos generados)
```

In [ ]:
#@title Ejercicio 01: Inicialización Git y estructura de directorios
# Crear estructura de directorios
base_path = Path('port_log')
dirs = [
    base_path / 'data' / 'raw',
    base_path / 'data' / 'interim',
    base_path / 'data' / 'interim' / 'plots',
    base_path / 'data' / 'processed',
    base_path / 'reports'
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f'Creado: {d}')

# Inicializar repositorio Git
!git init
!git config user.email "oscar.alejandrofranco@gmail.com"
!git config user.name "oscale05"
!git remote add origin https://github.com/oscale05/TRABAJO_PRACTICO_INTEGRADOR_UGR_HS.git
!git checkout -b Sprint_1

# Crear CHANGELOG.md inicial
changelog_content = """# CHANGELOG

## [Ejercicio 01] - Inicialización repositorio y estructura de directorios
- Inicializado repositorio Git en rama Sprint_1
- Configurado user.email y user.name
- Agregado remote origin
- Creada estructura de directorios port_log/data/{raw,interim,processed,interim/plots}, port_log/reports
- Commit inicial: \"Día 1: Inicialización repositorio y estructura de directorios\"
"""
with open('CHANGELOG.md', 'w') as f:
    f.write(changelog_content)

# Crear README.md inicial
readme_content = """# Trabajo Práctico Integrador - Port Log

## Sprint 1

### Objetivo
Aplicar conocimientos de versionado, organización y análisis exploratorio de datos con pandas sobre un dataset real de operaciones portuarias.

### Introducción y Contexto
El Puerto Fluvial de Rosario es uno de los complejos portuarios más importantes de América del Sur, siendo el principal punto de exportación de granos y derivados de la Argentina. Diariamente ingresan y egresan decenas de buques de distintas banderas con cargas de diverso tipo.

El sistema de registro de movimientos portuarios fue migrado recientemente desde un sistema heredado de los años '90. Ese sistema acumuló durante décadas inconsistencias de formato en fechas, matrículas de buques y valores numéricos fuera de rango, generando registros que no pueden incorporarse directamente al nuevo sistema.

Nuestro equipo fue contratado para analizar y depurar los datos del sistema antiguo.

Dataset: [port_movements](https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv)
"""
with open('README.md', 'w') as f:
    f.write(readme_content)

# Primer commit
!git add CHANGELOG.md README.md
!git commit -m "Día 1: Inicialización repositorio y estructura de directorios"

# Verificar
!git status
!git log --oneline -1

In [ ]:
#@title Posicionarse en directorio de trabajo y agregar la siguiente configuración a git
if DESACTIVAR_GIT_PUSH:
  !git config --global core.hooksPath {HOOKS}

#@title Ejercicio 02.1: Descargar dataset raw
import requests
from pathlib import Path

url = "https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv"
output_path = Path('port_log/data/raw/port_movements.csv')

response = requests.get(url)
response.raise_for_status()

with open(output_path, 'wb') as f:
    f.write(response.content)

print(f'Dataset descargado en: {output_path}')
print(f'Tamaño: {output_path.stat().st_size:,} bytes')

In [ ]:
#@title Ejercicio 02.1: Descargar dataset raw

url = "https://raw.githubusercontent.com/HAD141/datasets/refs/heads/main/TrabajosPracticos/port_log/port_movements.csv"
output_path = Path('port_log/data/raw/port_movements.csv')

response = requests.get(url)
response.raise_for_status()

with open(output_path, 'wb') as f:
    f.write(response.content)

print(f'Dataset descargado en: {output_path}')
print(f'Tamaño: {output_path.stat().st_size:,} bytes')

#@title Ejercicio 02.2: Mostrar 5 primeras y 5 últimas filas
import pandas as pd

df = pd.read_csv('port_log/data/raw/port_movements.csv')

# Concatenar head y tail en una única salida
result = pd.concat([df.head(5), df.tail(5)])
print(result.to_string())

In [ ]:
#@title Ejercicio 02.2: Mostrar 5 primeras y 5 últimas filas

df = pd.read_csv('port_log/data/raw/port_movements.csv')

# Concatenar head y tail en una única salida
result = pd.concat([df.head(5), df.tail(5)])
print(result.to_string())

#@title Ejercicio 02.3: Analizar tipos de datos e indicar conversiones
import pandas as pd

df = pd.read_csv('port_log/data/raw/port_movements.csv')

print("=== TIPOS DE DATOS ACTUALES ===")
print(df.dtypes.to_string())
print()
print("=== INFORMACIÓN DEL DATAFRAME ===")
print(df.info())
print()
print("=== COLUMNAS QUE REQUIEREN CONVERSIÓN ===")
print("• fecha_ingreso: object → datetime (formatos mixtos: YYYY-MM-DD, DD/MM/YYYY, DD-MM-YYYY)")
print("• fecha_egreso: object → datetime (formatos mixtos)")
print("• hora_ingreso: object → time (formatos mixtos: HH:MM, HH:MM AM/PM, inválidos como AB:CD)")
print("• hora_egreso: object → time (formatos mixtos)")
print("• tonelaje_declarado: object → float (tiene valores nulos)")
print("• velocidad_ingreso: float64 ✓")
print("• velocidad_maxima_muelle: int64 ✓")
print("• matricula: object → string (limpiar caracteres especiales, mayúsculas)")
print("• muelle: object → string (limpiar caracteres especiales, mayúsculas, normalizar espacios)")
print("• radar_id: object → string (tiene nulos)")
print("• estado_despacho: object → string (tiene nulos)")

In [ ]:
#@title Ejercicio 02.3: Analizar tipos de datos e indicar conversiones

df = pd.read_csv('port_log/data/raw/port_movements.csv')

print("=== TIPOS DE DATOS ACTUALES ===")
print(df.dtypes.to_string())
print()
print("=== INFORMACIÓN DEL DATAFRAME ===")
print(df.info())
print()
print("=== COLUMNAS QUE REQUIEREN CONVERSIÓN ===")
print("• fecha_ingreso: object → datetime (formatos mixtos: YYYY-MM-DD, DD/MM/YYYY, DD-MM-YYYY)")
print("• fecha_egreso: object → datetime (formatos mixtos)")
print("• hora_ingreso: object → time (formatos mixtos: HH:MM, HH:MM AM/PM, inválidos como AB:CD)")
print("• hora_egreso: object → time (formatos mixtos)")
print("• tonelaje_declarado: object → float (tiene valores nulos)")
print("• velocidad_ingreso: float64 ✓")
print("• velocidad_maxima_muelle: int64 ✓")
print("• matricula: object → string (limpiar caracteres especiales, mayúsculas)")
print("• muelle: object → string (limpiar caracteres especiales, mayúsculas, normalizar espacios)")
print("• radar_id: object → string (tiene nulos)")
print("• estado_despacho: object → string (tiene nulos)")

#@title Ejercicio 02.4: Contar valores nulos ordenados mayor a menor
import pandas as pd

df = pd.read_csv('port_log/data/raw/port_movements.csv')

null_counts = df.isnull().sum().sort_values(ascending=False)
null_df = pd.DataFrame({'columna': null_counts.index, 'nulos': null_counts.values})
null_df = null_df[null_df['nulos'] > 0]

print("=== VALORES NULOS POR COLUMNA (ordenado descendente) ===")
print(null_df.to_string(index=False))

In [ ]:
#@title Ejercicio 02.4: Contar valores nulos ordenados mayor a menor

df = pd.read_csv('port_log/data/raw/port_movements.csv')

null_counts = df.isnull().sum().sort_values(ascending=False)
null_df = pd.DataFrame({'columna': null_counts.index, 'nulos': null_counts.values})
null_df = null_df[null_df['nulos'] > 0]

print("=== VALORES NULOS POR COLUMNA (ordenado descendente) ===")
print(null_df.to_string(index=False))

#@title Ejercicio 02.5: Porcentaje de valores correctos por columna
import pandas as pd
import numpy as np

df = pd.read_csv('port_log/data/raw/port_movements.csv')

total_rows = len(df)
results = []

for col in df.columns:
    series = df[col]
    
    # Valores nulos = incorrectos
    null_count = series.isnull().sum()
    
    # Validar según tipo esperado
    if col in ['fecha_ingreso', 'fecha_egreso']:
        # Intentar parsear como fecha
        parsed = pd.to_datetime(series, errors='coerce', dayfirst=True)
        valid_count = parsed.notna().sum()
    elif col in ['hora_ingreso', 'hora_egreso']:
        # Validar formato de hora (HH:MM o HH:MM AM/PM)
        def is_valid_time(x):
            if pd.isna(x):
                return False
            x = str(x).strip()
            # Patrones válidos
            import re
            if re.match(r'^\d{1,2}:\d{2}$', x):
                h, m = map(int, x.split(':'))
                return 0 <= h <= 23 and 0 <= m <= 59
            if re.match(r'^\d{1,2}:\d{2}\s+(AM|PM)$', x, re.IGNORECASE):
                time_part = x[:-3].strip()
                h, m = map(int, time_part.split(':'))
                return 1 <= h <= 12 and 0 <= m <= 59
            return False
        valid_count = series.apply(is_valid_time).sum()
    elif col in ['tonelaje_declarado', 'velocidad_ingreso', 'velocidad_maxima_muelle']:
        # Deben ser numéricos válidos
        parsed = pd.to_numeric(series, errors='coerce')
        valid_count = parsed.notna().sum()
    elif col in ['matricula', 'muelle', 'radar_id', 'estado_despacho', 'tipo_carga', 'origen', 'buque_id', 'movimiento_id']:
        # Strings no nulos
        valid_count = series.notna().sum()
    else:
        valid_count = series.notna().sum()
    
    pct = (valid_count / total_rows) * 100
    results.append((col, pct))

print("Completitud del dataset:")
for col, pct in results:
    print(f" - {col}: {pct:.2f}% completa")

In [ ]:
#@title Ejercicio 02.5: Porcentaje de valores correctos por columna

df = pd.read_csv('port_log/data/raw/port_movements.csv')

total_rows = len(df)
results = []

for col in df.columns:
    series = df[col]
    
    # Valores nulos = incorrectos
    null_count = series.isnull().sum()
    
    # Validar según tipo esperado
    if col in ['fecha_ingreso', 'fecha_egreso']:
        # Intentar parsear como fecha
        parsed = pd.to_datetime(series, errors='coerce', dayfirst=True)
        valid_count = parsed.notna().sum()
    elif col in ['hora_ingreso', 'hora_egreso']:
        # Validar formato de hora (HH:MM o HH:MM AM/PM)
        def is_valid_time(x):
            if pd.isna(x):
                return False
            x = str(x).strip()
            # Patrones válidos
            if re.match(r'^\d{1,2}:\d{2}$', x):
                h, m = map(int, x.split(':'))
                return 0 <= h <= 23 and 0 <= m <= 59
            if re.match(r'^\d{1,2}:\d{2}\s+(AM|PM)$', x, re.IGNORECASE):
                time_part = x[:-3].strip()
                h, m = map(int, time_part.split(':'))
                return 1 <= h <= 12 and 0 <= m <= 59
            return False
        valid_count = series.apply(is_valid_time).sum()
    elif col in ['tonelaje_declarado', 'velocidad_ingreso', 'velocidad_maxima_muelle']:
        # Deben ser numéricos válidos
        parsed = pd.to_numeric(series, errors='coerce')
        valid_count = parsed.notna().sum()
    else:
        # Strings no nulos
        valid_count = series.notna().sum()
    
    pct = (valid_count / total_rows) * 100
    results.append((col, pct))

print("Completitud del dataset:")
for col, pct in results:
    print(f" - {col}: {pct:.2f}% completa")

## Ejercicio 03

> Puntos: 2

Utilizando Pandas, realizar la limpieza y normalización de datos:

- Normalizar fechas de ingreso y egreso al formato `YYYY-MM-DD`. Las fechas inválidas deben completarse con `1900-01-01`.

  Mostrar las 10 primeras fechas de ingreso.



In [ ]:
#@title Ejercicio 03.1: Normalizar fechas (YYYY-MM-DD), inválidas → 1900-01-01

df = pd.read_csv('port_log/data/raw/port_movements.csv')

# Función para parsear fechas con múltiples formatos
def parse_fecha(series):
    # Intentar dayfirst=True (DD/MM/YYYY o DD-MM-YYYY)
    parsed = pd.to_datetime(series, errors='coerce', dayfirst=True)
    # Las que no se pudieron parsear, intentar formato ISO (YYYY-MM-DD)
    mask_na = parsed.isna()
    if mask_na.any():
        parsed_iso = pd.to_datetime(series[mask_na], errors='coerce', dayfirst=False)
        parsed.loc[mask_na] = parsed_iso
    # Rellenar inválidas con 1900-01-01
    parsed = parsed.fillna(pd.Timestamp('1900-01-01'))
    return parsed.dt.strftime('%Y-%m-%d')

df['fecha_ingreso'] = parse_fecha(df['fecha_ingreso'])
df['fecha_egreso'] = parse_fecha(df['fecha_egreso'])

# Mostrar 10 primeras fechas de ingreso
print("=== 10 PRIMERAS FECHAS DE INGRESO (normalizadas) ===")
print(df['fecha_ingreso'].head(10).to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso1.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso1.csv")

#@title Ejercicio 03.1: Normalizar fechas (YYYY-MM-DD), inválidas → 1900-01-01
import pandas as pd
import numpy as np

df = pd.read_csv('port_log/data/raw/port_movements.csv')

# Función para parsear fechas con múltiples formatos
def parse_fecha(series):
    # Intentar dayfirst=True (DD/MM/YYYY o DD-MM-YYYY)
    parsed = pd.to_datetime(series, errors='coerce', dayfirst=True)
    # Las que no se pudieron parsear, intentar formato ISO (YYYY-MM-DD)
    mask_na = parsed.isna()
    if mask_na.any():
        parsed_iso = pd.to_datetime(series[mask_na], errors='coerce', dayfirst=False)
        parsed.loc[mask_na] = parsed_iso
    # Rellenar inválidas con 1900-01-01
    parsed = parsed.fillna(pd.Timestamp('1900-01-01'))
    return parsed.dt.strftime('%Y-%m-%d')

df['fecha_ingreso'] = parse_fecha(df['fecha_ingreso'])
df['fecha_egreso'] = parse_fecha(df['fecha_egreso'])

# Mostrar 10 primeras fechas de ingreso
print("=== 10 PRIMERAS FECHAS DE INGRESO (normalizadas) ===")
print(df['fecha_ingreso'].head(10).to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso1.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso1.csv")

In [ ]:
#@title Ejercicio 03.2: Normalizar horas a formato 24h, inválidas → 00:00

df = pd.read_csv('port_log/data/interim/port_movements_paso1.csv')

# Guardar originales para detectar inválidas
df['hora_ingreso_orig'] = df['hora_ingreso']
df['hora_egreso_orig'] = df['hora_egreso']

def parse_hora_24h(hora_str):
    """Convierte hora a formato HH:MM 24h. Retorna '00:00' si inválida."""
    if pd.isna(hora_str):
        return '00:00'
    hora_str = str(hora_str).strip()
    
    # Patrón 12h con AM/PM (ej: 10:55 PM, 12:22 AM, 06:54 PM)
    match_12h = re.match(r'^(\d{1,2}):(\d{2})\s*(AM|PM)$', hora_str, re.IGNORECASE)
    if match_12h:
        h, m, ampm = int(match_12h.group(1)), int(match_12h.group(2)), match_12h.group(3).upper()
        if not (1 <= h <= 12 and 0 <= m <= 59):
            return '00:00'
        if ampm == 'PM' and h != 12:
            h += 12
        elif ampm == 'AM' and h == 12:
            h = 0
        return f'{h:02d}:{m:02d}'
    
    # Patrón 24h (ej: 13:02, 5:50, 0:24, 21:20)
    match_24h = re.match(r'^(\d{1,2}):(\d{2})$', hora_str)
    if match_24h:
        h, m = int(match_24h.group(1)), int(match_24h.group(2))
        if 0 <= h <= 23 and 0 <= m <= 59:
            return f'{h:02d}:{m:02d}'
    
    # Cualquier otro formato → inválida
    return '00:00'

# Aplicar a ambas columnas
df['hora_ingreso'] = df['hora_ingreso'].apply(parse_hora_24h)
df['hora_egreso'] = df['hora_egreso'].apply(parse_hora_24h)

# Identificar horas inválidas
def era_invalida(original, normalizada):
    if pd.isna(original):
        return True
    original_str = str(original).strip()
    # Si la original ya era 00:00 o 0:00 válida, no es inválida
    if re.match(r'^0?0:00$', original_str):
        return False
    # Si la original era 12:00 AM → 00:00, no es inválida
    if re.match(r'^12:00\s*AM$', original_str, re.IGNORECASE):
        return False
    # Si normalizada es 00:00 y original no coincide con válidas
    if normalizada == '00:00':
        if re.match(r'^(\d{1,2}):(\d{2})(\s*(AM|PM))?$', original_str, re.IGNORECASE):
            h, m = map(int, original_str.split(':')[:2])
            if 'PM' in original_str.upper() and h != 12:
                return False
            if 'AM' in original_str.upper() and h == 12:
                return False
            if not ('AM' in original_str.upper() or 'PM' in original_str.upper()):
                if 0 <= h <= 23 and 0 <= m <= 59:
                    return False
        return True
    return False

invalidas_mask = df.apply(lambda r: era_invalida(r['hora_ingreso_orig'], r['hora_ingreso']), axis=1)
invalidas_ingreso = df[invalidas_mask].head(10)

print("=== 10 PRIMERAS HORAS DE INGRESO INVÁLIDAS (normalizadas a 00:00) ===")
if len(invalidas_ingreso) > 0:
    print(invalidas_ingreso[['movimiento_id', 'hora_ingreso_orig', 'hora_ingreso']].to_string(index=False))
else:
    print("No se detectaron horas inválidas en las primeras 10")

# Limpiar columnas auxiliares
df = df.drop(columns=['hora_ingreso_orig', 'hora_egreso_orig'])

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso2.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso2.csv")

#@title Ejercicio 03.2: Normalizar horas a formato 24h, inválidas → 00:00
import pandas as pd
import re

df = pd.read_csv('port_log/data/interim/port_movements_paso1.csv')

# Guardar originales para detectar inválidas
df['hora_ingreso_orig'] = df['hora_ingreso']
df['hora_egreso_orig'] = df['hora_egreso']

def parse_hora_24h(hora_str):
    """Convierte hora a formato HH:MM 24h. Retorna '00:00' si inválida."""
    if pd.isna(hora_str):
        return '00:00'
    hora_str = str(hora_str).strip()
    
    # Patrón 12h con AM/PM (ej: 10:55 PM, 12:22 AM, 06:54 PM)
    match_12h = re.match(r'^(\d{1,2}):(\d{2})\s*(AM|PM)$', hora_str, re.IGNORECASE)
    if match_12h:
        h, m, ampm = int(match_12h.group(1)), int(match_12h.group(2)), match_12h.group(3).upper()
        if not (1 <= h <= 12 and 0 <= m <= 59):
            return '00:00'
        if ampm == 'PM' and h != 12:
            h += 12
        elif ampm == 'AM' and h == 12:
            h = 0
        return f'{h:02d}:{m:02d}'
    
    # Patrón 24h (ej: 13:02, 5:50, 0:24, 21:20)
    match_24h = re.match(r'^(\d{1,2}):(\d{2})$', hora_str)
    if match_24h:
        h, m = int(match_24h.group(1)), int(match_24h.group(2))
        if 0 <= h <= 23 and 0 <= m <= 59:
            return f'{h:02d}:{m:02d}'
    
    # Cualquier otro formato → inválida
    return '00:00'

# Aplicar a ambas columnas
df['hora_ingreso'] = df['hora_ingreso'].apply(parse_hora_24h)
df['hora_egreso'] = df['hora_egreso'].apply(parse_hora_24h)

# Identificar horas inválidas
def era_invalida(original, normalizada):
    if pd.isna(original):
        return True
    original_str = str(original).strip()
    # Si la original ya era 00:00 o 0:00 válida, no es inválida
    if re.match(r'^0?0:00$', original_str):
        return False
    # Si la original era 12:00 AM → 00:00, no es inválida
    if re.match(r'^12:00\s*AM$', original_str, re.IGNORECASE):
        return False
    # Si normalizada es 00:00 y original no coincide con válidas
    if normalizada == '00:00':
        if re.match(r'^(\d{1,2}):(\d{2})(\s*(AM|PM))?$', original_str, re.IGNORECASE):
            h, m = map(int, original_str.split(':')[:2])
            if 'PM' in original_str.upper() and h != 12:
                return False
            if 'AM' in original_str.upper() and h == 12:
                return False
            if not ('AM' in original_str.upper() or 'PM' in original_str.upper()):
                if 0 <= h <= 23 and 0 <= m <= 59:
                    return False
        return True
    return False

invalidas_mask = df.apply(lambda r: era_invalida(r['hora_ingreso_orig'], r['hora_ingreso']), axis=1)
invalidas_ingreso = df[invalidas_mask].head(10)

print("=== 10 PRIMERAS HORAS DE INGRESO INVÁLIDAS (normalizadas a 00:00) ===")
if len(invalidas_ingreso) > 0:
    print(invalidas_ingreso[['movimiento_id', 'hora_ingreso_orig', 'hora_ingreso']].to_string(index=False))
else:
    print("No se detectaron horas inválidas en las primeras 10")

# Limpiar columnas auxiliares
df = df.drop(columns=['hora_ingreso_orig', 'hora_egreso_orig'])

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso2.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso2.csv")

In [ ]:
#@title Ejercicio 03.3: Calcular duracion_horas (diferencia ingreso-egreso)

df = pd.read_csv('port_log/data/interim/port_movements_paso2.csv')

# Combinar fecha + hora en datetime
def combine_datetime(fecha, hora):
    if fecha == '1900-01-01' or hora == '00:00':
        return pd.NaT
    try:
        return pd.to_datetime(fecha + ' ' + hora, format='%Y-%m-%d %H:%M')
    except:
        return pd.NaT

df['datetime_ingreso'] = df.apply(lambda r: combine_datetime(r['fecha_ingreso'], r['hora_ingreso']), axis=1)
df['datetime_egreso'] = df.apply(lambda r: combine_datetime(r['fecha_egreso'], r['hora_egreso']), axis=1)

# Calcular duración en horas
df['duracion_horas'] = (df['datetime_egreso'] - df['datetime_ingreso']).dt.total_seconds() / 3600

# Donde no se pudo calcular → pd.NA
df['duracion_horas'] = df['duracion_horas'].where(df['duracion_horas'].notna(), pd.NA)

# Mostrar 10 primeras duraciones
print("=== 10 PRIMERAS DURACIONES (horas) ===")
print(df[['movimiento_id', 'fecha_ingreso', 'hora_ingreso', 'fecha_egreso', 'hora_egreso', 'duracion_horas']].head(10).to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso3.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso3.csv")

#@title Ejercicio 03.3: Calcular duracion_horas (diferencia ingreso-egreso)
import pandas as pd
import numpy as np

df = pd.read_csv('port_log/data/interim/port_movements_paso2.csv')

# Combinar fecha + hora en datetime
def combine_datetime(fecha, hora):
    if fecha == '1900-01-01' or hora == '00:00':
        return pd.NaT
    try:
        return pd.to_datetime(fecha + ' ' + hora, format='%Y-%m-%d %H:%M')
    except:
        return pd.NaT

df['datetime_ingreso'] = df.apply(lambda r: combine_datetime(r['fecha_ingreso'], r['hora_ingreso']), axis=1)
df['datetime_egreso'] = df.apply(lambda r: combine_datetime(r['fecha_egreso'], r['hora_egreso']), axis=1)

# Calcular duración en horas
df['duracion_horas'] = (df['datetime_egreso'] - df['datetime_ingreso']).dt.total_seconds() / 3600

# Donde no se pudo calcular → pd.NA
df['duracion_horas'] = df['duracion_horas'].where(df['duracion_horas'].notna(), pd.NA)

# Mostrar 10 primeras duraciones
print("=== 10 PRIMERAS DURACIONES (horas) ===")
print(df[['movimiento_id', 'fecha_ingreso', 'hora_ingreso', 'fecha_egreso', 'hora_egreso', 'duracion_horas']].head(10).to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso3.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso3.csv")

In [ ]:
#@title Ejercicio 03.4: Normalizar matrículas (quitar especiales, mayúsculas, inválidas → pd.NA)

df = pd.read_csv('port_log/data/interim/port_movements_paso3.csv')

def normalizar_matricula(mat):
    if pd.isna(mat):
        return pd.NA
    # Convertir a string y mayúsculas
    mat = str(mat).upper().strip()
    # Quitar caracteres especiales (mantener solo alfanuméricos y guión)
    mat_clean = re.sub(r'[^A-Z0-9-]', '', mat)
    # Validar: debe tener al menos un carácter alfanumérico
    if not re.match(r'^[A-Z0-9-]+$', mat_clean) or len(mat_clean) == 0:
        return pd.NA
    # No puede ser solo guiones
    if mat_clean.replace('-', '') == '':
        return pd.NA
    return mat_clean

df['matricula'] = df['matricula'].apply(normalizar_matricula)

# Mostrar 10 últimas matrículas
print("=== 10 ÚLTIMAS MATRÍCULAS (normalizadas) ===")
print(df['matricula'].tail(10).to_string(index=False))

# Estadísticas
n_na = df['matricula'].isna().sum()
print(f"\nMatrículas nulas/NA después de normalizar: {n_na}")

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso4.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso4.csv")

#@title Ejercicio 03.4: Normalizar matrículas (quitar especiales, mayúsculas, inválidas → pd.NA)
import pandas as pd
import re

df = pd.read_csv('port_log/data/interim/port_movements_paso3.csv')

def normalizar_matricula(mat):
    if pd.isna(mat):
        return pd.NA
    # Convertir a string y mayúsculas
    mat = str(mat).upper().strip()
    # Quitar caracteres especiales (mantener solo alfanuméricos y guión)
    mat_clean = re.sub(r'[^A-Z0-9-]', '', mat)
    # Validar: debe tener al menos un carácter alfanumérico
    if not re.match(r'^[A-Z0-9-]+$', mat_clean) or len(mat_clean) == 0:
        return pd.NA
    # No puede ser solo guiones
    if mat_clean.replace('-', '') == '':
        return pd.NA
    return mat_clean

df['matricula'] = df['matricula'].apply(normalizar_matricula)

# Mostrar 10 últimas matrículas
print("=== 10 ÚLTIMAS MATRÍCULAS (normalizadas) ===")
print(df['matricula'].tail(10).to_string(index=False))

# Estadísticas
n_na = df['matricula'].isna().sum()
print(f"\nMatrículas nulas/NA después de normalizar: {n_na}")

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso4.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso4.csv")

In [ ]:
#@title Ejercicio 03.5: Normalizar muelles (quitar especiales, mayúsculas)

df = pd.read_csv('port_log/data/interim/port_movements_paso4.csv')

def normalizar_muelle(muelle):
    if pd.isna(muelle):
        return pd.NA
    muelle = str(muelle).upper().strip()
    # Quitar caracteres especiales, normalizar espacios
    muelle = re.sub(r'[^A-Z0-9\s-]', '', muelle)
    muelle = re.sub(r'\s+', ' ', muelle).strip()
    # Estandarizar formato MUELLE-X
    muelle = muelle.replace(' ', '-')
    return muelle

df['muelle'] = df['muelle'].apply(normalizar_muelle)

# Mostrar 10 primeros muelles
print("=== 10 PRIMEROS MUELLES (normalizados) ===")
print(df['muelle'].head(10).to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso5.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso5.csv")

#@title Ejercicio 03.5: Normalizar muelles (quitar especiales, mayúsculas)
import pandas as pd
import re

df = pd.read_csv('port_log/data/interim/port_movements_paso4.csv')

def normalizar_muelle(muelle):
    if pd.isna(muelle):
        return pd.NA
    muelle = str(muelle).upper().strip()
    # Quitar caracteres especiales, normalizar espacios
    muelle = re.sub(r'[^A-Z0-9\s-]', '', muelle)
    muelle = re.sub(r'\s+', ' ', muelle).strip()
    # Estandarizar formato MUELLE-X
    muelle = muelle.replace(' ', '-')
    return muelle

df['muelle'] = df['muelle'].apply(normalizar_muelle)

# Mostrar 10 primeros muelles
print("=== 10 PRIMEROS MUELLES (normalizados) ===")
print(df['muelle'].head(10).to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso5.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso5.csv")

In [ ]:
#@title Ejercicio 03.6: Eliminar filas con nulos en columnas críticas

df = pd.read_csv('port_log/data/interim/port_movements_paso5.csv')

# Columnas críticas seleccionadas:
# - matricula: identifica al buque, esencial para análisis de infracciones
# - fecha_ingreso / fecha_egreso: necesarias para calcular duración y análisis temporal
# - muelle: clave para análisis por muelle y exceso por muelle
# - velocidad_ingreso / velocidad_maxima_muelle: base para calcular excesos de velocidad
# - tipo_carga / origen: variables de análisis en ejercicios posteriores
columnas_criticas = [
    'matricula',
    'fecha_ingreso',
    'fecha_egreso',
    'muelle',
    'velocidad_ingreso',
    'velocidad_maxima_muelle',
    'tipo_carga',
    'origen'
]

filas_antes = len(df)
df_clean = df.dropna(subset=columnas_criticas).copy()
filas_despues = len(df_clean)
filas_eliminadas = filas_antes - filas_despues

# Contar eliminadas por columna (aproximado: filas que tenían nulo en esa columna)
print("Las columnas críticas son:")
for col in columnas_criticas:
    nulos_en_col = df[col].isna().sum()
    print(f" - {col}: {nulos_en_col}")

print(f"\nFilas totales antes: {filas_antes}")
print(f"Filas totales después: {filas_despues}")
print(f"Filas eliminadas: {filas_eliminadas}")

# Guardar progreso
df_clean.to_csv('port_log/data/interim/port_movements_paso6.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso6.csv")

#@title Ejercicio 03.6: Eliminar filas con nulos en columnas críticas
import pandas as pd

df = pd.read_csv('port_log/data/interim/port_movements_paso5.csv')

# Columnas críticas seleccionadas:
# - matricula: identifica al buque, esencial para análisis de infracciones
# - fecha_ingreso / fecha_egreso: necesarias para calcular duración y análisis temporal
# - muelle: clave para análisis por muelle y exceso por muelle
# - velocidad_ingreso / velocidad_maxima_muelle: base para calcular excesos de velocidad
# - tipo_carga / origen: variables de análisis en ejercicios posteriores
columnas_criticas = [
    'matricula',
    'fecha_ingreso',
    'fecha_egreso',
    'muelle',
    'velocidad_ingreso',
    'velocidad_maxima_muelle',
    'tipo_carga',
    'origen'
]

filas_antes = len(df)
df_clean = df.dropna(subset=columnas_criticas).copy()
filas_despues = len(df_clean)
filas_eliminadas = filas_antes - filas_despues

# Contar eliminadas por columna (aproximado: filas que tenían nulo en esa columna)
print("Las columnas críticas son:")
for col in columnas_criticas:
    nulos_en_col = df[col].isna().sum()
    print(f" - {col}: {nulos_en_col}")

print(f"\nFilas totales antes: {filas_antes}")
print(f"Filas totales después: {filas_despues}")
print(f"Filas eliminadas: {filas_eliminadas}")

# Guardar progreso
df_clean.to_csv('port_log/data/interim/port_movements_paso6.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso6.csv")

In [ ]:
#@title Ejercicio 03.7: Detectar y eliminar outliers (IQR vs Z-score) en tonelaje_declarado y velocidad_ingreso

df = pd.read_csv('port_log/data/interim/port_movements_paso6.csv')

def detectar_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

def detectar_outliers_zscore(series, threshold=3):
    mean = series.mean()
    std = series.std()
    if std == 0:
        return pd.Series(False, index=series.index)
    z_scores = np.abs((series - mean) / std)
    return z_scores > threshold

columnas_outliers = ['tonelaje_declarado', 'velocidad_ingreso']
filas_eliminadas_total = 0

print("=== COMPARACIÓN IQR vs Z-SCORE ===\n")

for col in columnas_outliers:
    series = df[col].dropna()
    
    outliers_iqr = detectar_outliers_iqr(series)
    outliers_zscore = detectar_outliers_zscore(series)
    
    n_iqr = outliers_iqr.sum()
    n_zscore = outliers_zscore.sum()
    
    print(f"--- {col} ---")
    print(f"  IQR (1.5*IQR): {n_iqr} outliers detectados")
    print(f"  Z-score (>3):  {n_zscore} outliers detectados")
    
    # JUSTIFICACIÓN: Usamos IQR porque:
    # 1. No asume distribución normal (los datos portuarios son sesgados)
    # 2. Es más robusto ante valores extremos
    # 3. Z-score es sensible a los propios outliers (inflan media y std)
    print(f"  -> MÉTODO ELEGIDO: IQR (más robusto para datos no normales)")
    print()
    
    # Aplicar IQR
    mask_outliers = detectar_outliers_iqr(df[col].dropna())
    indices_outliers = df[col].dropna().index[mask_outliers]
    df = df.drop(index=indices_outliers)
    filas_eliminadas_total += len(indices_outliers)

print(f"Total filas eliminadas por outliers (IQR): {filas_eliminadas_total}")
print(f"Filas restantes: {len(df)}")

print("\nLas columnas críticas son:")
for col in columnas_outliers:
    n_outliers = detectar_outliers_iqr(df[col].dropna()).sum()
    print(f" - {col}: {n_outliers} (después de limpieza)")

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso7.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso7.csv")

#@title Ejercicio 03.7: Detectar y eliminar outliers (IQR vs Z-score) en tonelaje_declarado y velocidad_ingreso
import pandas as pd
import numpy as np

df = pd.read_csv('port_log/data/interim/port_movements_paso6.csv')

def detectar_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

def detectar_outliers_zscore(series, threshold=3):
    mean = series.mean()
    std = series.std()
    if std == 0:
        return pd.Series(False, index=series.index)
    z_scores = np.abs((series - mean) / std)
    return z_scores > threshold

columnas_outliers = ['tonelaje_declarado', 'velocidad_ingreso']
filas_eliminadas_total = 0

print("=== COMPARACIÓN IQR vs Z-SCORE ===\n")

for col in columnas_outliers:
    series = df[col].dropna()
    
    outliers_iqr = detectar_outliers_iqr(series)
    outliers_zscore = detectar_outliers_zscore(series)
    
    n_iqr = outliers_iqr.sum()
    n_zscore = outliers_zscore.sum()
    
    print(f"--- {col} ---")
    print(f"  IQR (1.5*IQR): {n_iqr} outliers detectados")
    print(f"  Z-score (>3):  {n_zscore} outliers detectados")
    
    # JUSTIFICACIÓN: Usamos IQR porque:
    # 1. No asume distribución normal (los datos portuarios son sesgados)
    # 2. Es más robusto ante valores extremos
    # 3. Z-score es sensible a los propios outliers (inflan media y std)
    print(f"  -> MÉTODO ELEGIDO: IQR (más robusto para datos no normales)")
    print()
    
    # Aplicar IQR
    mask_outliers = detectar_outliers_iqr(df[col].dropna())
    indices_outliers = df[col].dropna().index[mask_outliers]
    df = df.drop(index=indices_outliers)
    filas_eliminadas_total += len(indices_outliers)

print(f"Total filas eliminadas por outliers (IQR): {filas_eliminadas_total}")
print(f"Filas restantes: {len(df)}")

print("\nLas columnas críticas son:")
for col in columnas_outliers:
    n_outliers = detectar_outliers_iqr(df[col].dropna()).sum()
    print(f" - {col}: {n_outliers} (después de limpieza)")

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso7.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso7.csv")

In [ ]:
#@title Ejercicio 03.8: Crear exceso_velocidad_real = velocidad_ingreso - velocidad_maxima_muelle

df = pd.read_csv('port_log/data/interim/port_movements_paso7.csv')

df['exceso_velocidad_real'] = df['velocidad_ingreso'] - df['velocidad_maxima_muelle']

# Mostrar 10 primeras matrículas con su valor
print("=== 10 PRIMERAS MATRÍCULAS CON EXCESO_VELOCIDAD_REAL ===")
result = df[['matricula', 'exceso_velocidad_real']].head(10)
print(result.to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso8.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso8.csv")

#@title Ejercicio 03.8: Crear exceso_velocidad_real = velocidad_ingreso - velocidad_maxima_muelle
import pandas as pd

df = pd.read_csv('port_log/data/interim/port_movements_paso7.csv')

df['exceso_velocidad_real'] = df['velocidad_ingreso'] - df['velocidad_maxima_muelle']

# Mostrar 10 primeras matrículas con su valor
print("=== 10 PRIMERAS MATRÍCULAS CON EXCESO_VELOCIDAD_REAL ===")
result = df[['matricula', 'exceso_velocidad_real']].head(10)
print(result.to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso8.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso8.csv")

In [ ]:
#@title Ejercicio 03.9: Crear exceso_velocidad = velocidad_ingreso - (velocidad_maxima_muelle * 1.05)

df = pd.read_csv('port_log/data/interim/port_movements_paso8.csv')

# 5% de tolerancia sobre velocidad_maxima_muelle
df['exceso_velocidad'] = df['velocidad_ingreso'] - (df['velocidad_maxima_muelle'] * 1.05)

# Mostrar 10 primeras matrículas con su valor
print("=== 10 PRIMERAS MATRÍCULAS CON EXCESO_VELOCIDAD (con 5% tolerancia) ===")
result = df[['matricula', 'exceso_velocidad']].head(10)
print(result.to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso9.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso9.csv")

#@title Ejercicio 03.9: Crear exceso_velocidad = velocidad_ingreso - (velocidad_maxima_muelle * 1.05)
import pandas as pd

df = pd.read_csv('port_log/data/interim/port_movements_paso8.csv')

# 5% de tolerancia sobre velocidad_maxima_muelle
df['exceso_velocidad'] = df['velocidad_ingreso'] - (df['velocidad_maxima_muelle'] * 1.05)

# Mostrar 10 primeras matrículas con su valor
print("=== 10 PRIMERAS MATRÍCULAS CON EXCESO_VELOCIDAD (con 5% tolerancia) ===")
result = df[['matricula', 'exceso_velocidad']].head(10)
print(result.to_string(index=False))

# Guardar progreso
df.to_csv('port_log/data/interim/port_movements_paso9.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso9.csv")

In [ ]:
#@title Ejercicio 03.10: Eliminar filas sin infracción (exceso_velocidad <= 0)

df = pd.read_csv('port_log/data/interim/port_movements_paso9.csv')

filas_antes = len(df)
df_infracciones = df[df['exceso_velocidad'] > 0].copy()
filas_despues = len(df_infracciones)
filas_eliminadas = filas_antes - filas_despues

msg = f"Se eliminaron {filas_eliminadas} filas sin infracción."
print(msg)
print(f"Filas antes: {filas_antes}, Filas después: {filas_despues}")

# Guardar progreso
df_infracciones.to_csv('port_log/data/interim/port_movements_paso10.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso10.csv")

#@title Ejercicio 03.10: Eliminar filas sin infracción (exceso_velocidad <= 0)
import pandas as pd

df = pd.read_csv('port_log/data/interim/port_movements_paso9.csv')

filas_antes = len(df)
df_infracciones = df[df['exceso_velocidad'] > 0].copy()
filas_despues = len(df_infracciones)
filas_eliminadas = filas_antes - filas_despues

msg = f"Se eliminaron {filas_eliminadas} filas sin infracción."
print(msg)
print(f"Filas antes: {filas_antes}, Filas después: {filas_despues}")

# Guardar progreso
df_infracciones.to_csv('port_log/data/interim/port_movements_paso10.csv', index=False)
print("\nGuardado en port_log/data/interim/port_movements_paso10.csv")

In [ ]:
#@title Ejercicio 03.11: Guardar dataset limpio final en interim

df = pd.read_csv('port_log/data/interim/port_movements_paso10.csv')

output_path = Path('port_log/data/interim/port_movements.csv')
df.to_csv(output_path, index=False)

print(f"Dataset limpio guardado en: {output_path}")
print(f"Filas: {len(df)}, Columnas: {len(df.columns)}")
print(f"Columnas: {list(df.columns)}")

#@title Ejercicio 03.11: Guardar dataset limpio final en interim
import pandas as pd
from pathlib import Path

df = pd.read_csv('port_log/data/interim/port_movements_paso10.csv')

output_path = Path('port_log/data/interim/port_movements.csv')
df.to_csv(output_path, index=False)

print(f"Dataset limpio guardado en: {output_path}")
print(f"Filas: {len(df)}, Columnas: {len(df.columns)}")
print(f"Columnas: {list(df.columns)}")

In [ ]:
#@title Ejercicio 03.12: Exportar resumen estadístico (describe) a reports/summary_sprint1.csv

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# describe() con include='all' para todas las columnas
summary = df.describe(include='all')

output_path = Path('port_log/reports/summary_sprint1.csv')
summary.to_csv(output_path)

print(f"Resumen estadístico guardado en: {output_path}")
print(summary.to_string())

#@title Ejercicio 03.12: Exportar resumen estadístico (describe) a reports/summary_sprint1.csv
import pandas as pd
from pathlib import Path

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# describe() con include='all' para todas las columnas
summary = df.describe(include='all')

output_path = Path('port_log/reports/summary_sprint1.csv')
summary.to_csv(output_path)

print(f"Resumen estadístico guardado en: {output_path}")
print(summary.to_string())

## Ejercicio 04

> Puntos: 2

Definir una clase `PortAnalyzer` que:

- Reciba el DataFrame limpio en `__init__` y encapsule los datos correctamente.

- Implemente los siguientes métodos:

  - **`top_infractores(n: int) -> pd.DataFrame`**: ranking de las `n` matrículas con más infracciones.
  
    Retorna un DataFrame ordenado de mayor a menor, índice desde 1, columnas `matricula` y `cantidad`.

  - **`infracciones_por_turno() -> pd.DataFrame`**: agrupa infracciones en turnos del día según la hora de ingreso:
    - Madrugada: 00:00–05:59
    - Mañana: 06:00–11:59
    - Tarde: 12:00–17:59
    - Noche: 18:00–23:59
    
    Retorna un DataFrame con columnas `turno` y `cantidad`, ordenado de mayor a menor.

  - **`exceso_promedio() -> float`**: Retorna el exceso de velocidad promedio calculado sobre la columna `exceso_velocidad_real`.

  - **`exceso_promedio_tolerancia() -> float`**: Retorna el exceso de velocidad promedio calculado sobre la columna `exceso_velocidad`.

  - **`infracciones_por_muelle() -> pd.DataFrame`**: Retorna un DataFrame con la cantidad de infracciones por muelle, ordenado alfabéticamente con las columnas `muelle` y `cantidad`.

  - **`infractores_por_tipo_carga() -> pd.DataFrame`**: Retorna un DataFrame con la cantidad de infracciones agrupadas por `tipo_carga`, de mayor a menor con las columnas `tipo_carga` y `cantidad`.


In [ ]:
#@title Ejercicio 04: Definir clase PortAnalyzer

class PortAnalyzer:
    """Analizador de infracciones portuarias."""
    
    def __init__(self, df: pd.DataFrame) -> None:
        """Inicializa con el DataFrame limpio, encapsulando los datos."""
        self._df = df.copy()
    
    def top_infractores(self, n: int) -> pd.DataFrame:
        """Ranking de las n matrículas con más infracciones."""
        ranking = (
            self._df.groupby('matricula')
            .size()
            .reset_index(name='cantidad')
            .sort_values('cantidad', ascending=False)
            .head(n)
            .reset_index(drop=True)
        )
        ranking.index = ranking.index + 1
        return ranking[['matricula', 'cantidad']]
    
    def infracciones_por_turno(self) -> pd.DataFrame:
        """Agrupa infracciones por turno del día según hora_ingreso."""
        def hora_a_turno(hora_str: str) -> str:
            if pd.isna(hora_str) or hora_str == '00:00':
                return 'Desconocido'
            h = int(str(hora_str).split(':')[0])
            if 0 <= h <= 5:
                return 'Madrugada'
            elif 6 <= h <= 11:
                return 'Mañana'
            elif 12 <= h <= 17:
                return 'Tarde'
            else:
                return 'Noche'
        
        df = self._df.copy()
        df['turno'] = df['hora_ingreso'].apply(hora_a_turno)
        
        result = (
            df.groupby('turno')
            .size()
            .reset_index(name='cantidad')
            .sort_values('cantidad', ascending=False)
            .reset_index(drop=True)
        )
        return result[['turno', 'cantidad']]
    
    def exceso_promedio(self) -> float:
        """Exceso de velocidad promedio (exceso_velocidad_real)."""
        return float(self._df['exceso_velocidad_real'].mean())
    
    def exceso_promedio_tolerancia(self) -> float:
        """Exceso de velocidad promedio con tolerancia (exceso_velocidad)."""
        return float(self._df['exceso_velocidad'].mean())
    
    def infracciones_por_muelle(self) -> pd.DataFrame:
        """Cantidad de infracciones por muelle, ordenado alfabéticamente."""
        result = (
            self._df.groupby('muelle')
            .size()
            .reset_index(name='cantidad')
            .sort_values('muelle', ascending=True)
            .reset_index(drop=True)
        )
        return result[['muelle', 'cantidad']]
    
    def infractores_por_tipo_carga(self) -> pd.DataFrame:
        """Cantidad de infracciones por tipo_carga, ordenado mayor a menor."""
        result = (
            self._df.groupby('tipo_carga')
            .size()
            .reset_index(name='cantidad')
            .sort_values('cantidad', ascending=False)
            .reset_index(drop=True)
        )
        return result[['tipo_carga', 'cantidad']]

- Crear el objeto `PortAnalyzer` e invocar cada método en celdas separadas.

In [ ]:
#@title Ejercicio 04: Instanciar PortAnalyzer e invocar métodos

df = pd.read_csv('port_log/data/interim/port_movements.csv')

analyzer = PortAnalyzer(df)

print("=== PortAnalyzer instanciado ===")
print(f"Total registros: {len(df)}")
print()

In [ ]:
#@title Ejercicio 04: top_infractores(10)
print("=== TOP 10 INFRACTORES ===")
result = analyzer.top_infractores(10)
print(result.to_string(index=True))

In [ ]:
#@title Ejercicio 04: infracciones_por_turno
print("=== INFRACCIONES POR TURNO ===")
result = analyzer.infracciones_por_turno()
print(result.to_string(index=False))

In [ ]:
#@title Ejercicio 04: exceso_promedio
print("=== EXCESO PROMEDIO (real) ===")
result = analyzer.exceso_promedio()
print(f"{result:.4f}")

In [ ]:
#@title Ejercicio 04: exceso_promedio_tolerancia
print("=== EXCESO PROMEDIO (con 5% tolerancia) ===")
result = analyzer.exceso_promedio_tolerancia()
print(f"{result:.4f}")

In [ ]:
#@title Ejercicio 04: infracciones_por_muelle
print("=== INFRACCIONES POR MUELLE ===")
result = analyzer.infracciones_por_muelle()
print(result.to_string(index=False))

In [ ]:
#@title Ejercicio 04: infractores_por_tipo_carga
print("=== INFRACTORES POR TIPO DE CARGA ===")
result = analyzer.infractores_por_tipo_carga()
print(result.to_string(index=False))

## Ejercicio 05

> Puntos: 2

Responder cada punto con un gráfico. Cada gráfico se debe mostrar en el notebook. Todos deben tener título, etiquetas en los ejes y leyenda cuando corresponda.

- **Top 10 matrículas más reincidentes** en barras verticales, de mayor a menor.

  Exportar: `port_log/data/interim/plots/top_infractores.jpg`.



In [ ]:
#@title Ejercicio 05.1: Top 10 matrículas más reincidentes (barras verticales)

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# Obtener top 10
top10 = df['matricula'].value_counts().head(10).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(range(len(top10)), top10.values, color='steelblue', edgecolor='black')
plt.yticks(range(len(top10)), top10.index)
plt.xlabel('Cantidad de infracciones')
plt.ylabel('Matrícula')
plt.title('Top 10 Matrículas Más Reincidentes', fontsize=14, fontweight='bold')
plt.tight_layout()

# Agregar valores en las barras
for i, (idx, val) in enumerate(top10.items()):
    plt.text(val + 0.5, i, str(val), va='center', fontweight='bold')

output_path = Path('port_log/data/interim/plots/top_infractores.jpg')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Gráfico guardado en: {output_path}')
plt.show()

- **Total de infracciones por turno del día** (Madrugada / Mañana / Tarde / Noche) en gráfico de torta.

  Exportar: `port_log/data/interim/plots/turnos.jpg`.


In [ ]:
#@title Ejercicio 05.2: Infracciones por turno (gráfico de torta)

df = pd.read_csv('port_log/data/interim/port_movements.csv')

def hora_a_turno(hora_str):
    if pd.isna(hora_str) or hora_str == '00:00':
        return 'Desconocido'
    h = int(str(hora_str).split(':')[0])
    if 0 <= h <= 5:
        return 'Madrugada'
    elif 6 <= h <= 11:
        return 'Mañana'
    elif 12 <= h <= 17:
        return 'Tarde'
    else:
        return 'Noche'

df['turno'] = df['hora_ingreso'].apply(hora_a_turno)
turno_counts = df['turno'].value_counts()

# Orden específico para la leyenda
orden_turnos = ['Madrugada', 'Mañana', 'Tarde', 'Noche', 'Desconocido']
turno_counts = turno_counts.reindex([t for t in orden_turnos if t in turno_counts.index])

plt.figure(figsize=(8, 8))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
wedges, texts, autotexts = plt.pie(
    turno_counts.values,
    labels=turno_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors[:len(turno_counts)],
    textprops={'fontsize': 11}
)
plt.title('Total de Infracciones por Turno del Día', fontsize=14, fontweight='bold')
plt.tight_layout()

output_path = Path('port_log/data/interim/plots/turnos.jpg')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Gráfico guardado en: {output_path}')
plt.show()


- **Total de infracciones por mes** en barras horizontal, ordenado de mayor a menor.

  Exportar: `port_log/data/interim/plots/meses.jpg`.


In [ ]:
#@title Ejercicio 05.3: Infracciones por mes (barras horizontales)

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# Extraer mes de fecha_ingreso
df['mes'] = pd.to_datetime(df['fecha_ingreso'], errors='coerce').dt.month
mes_counts = df['mes'].value_counts().sort_index()

# Nombres de meses
meses_nombres = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
mes_labels = [meses_nombres[m-1] for m in mes_counts.index]

# Ordenar de mayor a menor
sorted_data = sorted(zip(mes_labels, mes_counts.values), key=lambda x: x[1], reverse=True)
labels_sorted, values_sorted = zip(*sorted_data)

plt.figure(figsize=(10, 6))
bars = plt.barh(range(len(values_sorted)), values_sorted, color='coral', edgecolor='black')
plt.yticks(range(len(values_sorted)), labels_sorted)
plt.xlabel('Cantidad de infracciones')
plt.ylabel('Mes')
plt.title('Total de Infracciones por Mes', fontsize=14, fontweight='bold')
plt.tight_layout()

for i, val in enumerate(values_sorted):
    plt.text(val + 0.5, i, str(val), va='center', fontweight='bold')

output_path = Path('port_log/data/interim/plots/meses.jpg')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Gráfico guardado en: {output_path}')
plt.show()


- **Histograma del exceso de velocidad real** con curva de densidad (KDE) superpuesta.

  Exportar: `port_log/data/interim/plots/distribucion_exceso.jpg`.


In [ ]:
#@title Ejercicio 05.4: Histograma exceso_velocidad_real con KDE

df = pd.read_csv('port_log/data/interim/port_movements.csv')

exceso = df['exceso_velocidad_real'].dropna()

plt.figure(figsize=(10, 6))

# Histograma
n, bins, patches = plt.hist(exceso, bins=30, density=True, alpha=0.6, color='skyblue', edgecolor='black', label='Histograma')

# KDE
kde = gaussian_kde(exceso)
x_range = np.linspace(exceso.min(), exceso.max(), 200)
plt.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE (Densidad)')

plt.xlabel('Exceso de velocidad real')
plt.ylabel('Densidad')
plt.title('Histograma del Exceso de Velocidad Real con Curva de Densidad (KDE)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

output_path = Path('port_log/data/interim/plots/distribucion_exceso.jpg')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Gráfico guardado en: {output_path}')
plt.show()


- **Exceso de velocidad promedio por muelle** en barras horizontal, ordenado de mayor a menor.

  Exportar: `port_log/data/interim/plots/exceso_por_muelle.jpg`.


In [ ]:
#@title Ejercicio 05.5: Exceso de velocidad promedio por muelle (barras horizontales)

df = pd.read_csv('port_log/data/interim/port_movements.csv')

exceso_muelle = df.groupby('muelle')['exceso_velocidad_real'].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(range(len(exceso_muelle)), exceso_muelle.values, color='mediumseagreen', edgecolor='black')
plt.yticks(range(len(exceso_muelle)), exceso_muelle.index)
plt.xlabel('Exceso de velocidad promedio (real)')
plt.ylabel('Muelle')
plt.title('Exceso de Velocidad Promedio por Muelle', fontsize=14, fontweight='bold')
plt.tight_layout()

for i, val in enumerate(exceso_muelle.values):
    plt.text(val + 0.05, i, f'{val:.2f}', va='center', fontweight='bold')

output_path = Path('port_log/data/interim/plots/exceso_por_muelle.jpg')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Gráfico guardado en: {output_path}')
plt.show()


- **Comparación de infracciones con fecha válida vs fecha inválida (`1900-01-01`)** en barras simple.

  Exportar: `port_log/data/interim/plots/fechas_invalidas.jpg`.

In [ ]:
#@title Ejercicio 05.6: Fechas válidas vs inválidas (barras simples)

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# Contar fechas válidas vs inválidas (1900-01-01)
fecha_invalida = (df['fecha_ingreso'] == '1900-01-01') | (df['fecha_egreso'] == '1900-01-01')
n_invalidas = fecha_invalida.sum()
n_validas = len(df) - n_invalidas

plt.figure(figsize=(6, 5))
categorias = ['Fecha válida', 'Fecha inválida (1900-01-01)']
valores = [n_validas, n_invalidas]
colores = ['seagreen', 'indianred']

bars = plt.bar(categorias, valores, color=colores, edgecolor='black', width=0.5)
plt.ylabel('Cantidad de registros')
plt.title('Comparación: Fechas Válidas vs Inválidas', fontsize=14, fontweight='bold')
plt.tight_layout()

for bar, val in zip(bars, valores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
             ha='center', va='bottom', fontweight='bold', fontsize=12)

output_path = Path('port_log/data/interim/plots/fechas_invalidas.jpg')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Gráfico guardado en: {output_path}')
plt.show()

## Ejercicio 06

> Puntos: 1

Responder las siguientes preguntas con el dataset limpio:

- ¿Qué porcentaje de infracciones provienen de registros con fecha inválida?
```
El porcentaje de infracciones con fecha inválida es XX.XX%
```



In [ ]:
#@title Ejercicio 06.1: % infracciones con fecha inválida

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# Fecha inválida = 1900-01-01 en fecha_ingreso O fecha_egreso
fecha_invalida = (df['fecha_ingreso'] == '1900-01-01') | (df['fecha_egreso'] == '1900-01-01')
n_invalidas = fecha_invalida.sum()
total = len(df)
pct = (n_invalidas / total) * 100

print(f"El porcentaje de infracciones con fecha inválida es {pct:.2f}%")

- ¿Qué porcentaje de infracciones provienen de registros con hora inválida?
```
El porcentaje de infracciones con hora inválida es XX.XX%
```


In [ ]:
#@title Ejercicio 06.2: % infracciones con hora inválida

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# Hora inválida = 00:00 en hora_ingreso O hora_egreso
hora_invalida = (df['hora_ingreso'] == '00:00') | (df['hora_egreso'] == '00:00')
n_invalidas = hora_invalida.sum()
total = len(df)
pct = (n_invalidas / total) * 100

print(f"El porcentaje de infracciones con hora inválida es {pct:.2f}%")


- ¿Cuál es el tipo de carga más frecuente en infracciones y qué porcentaje representa?
```
El tipo de carga más frecuente es XXXX con XX.XX%
```


In [ ]:
#@title Ejercicio 06.3: Tipo de carga más frecuente en infracciones

df = pd.read_csv('port_log/data/interim/port_movements.csv')

tipo_counts = df['tipo_carga'].value_counts()
tipo_top = tipo_counts.index[0]
tipo_count = tipo_counts.iloc[0]
total = len(df)
pct = (tipo_count / total) * 100

print(f"El tipo de carga más frecuente es {tipo_top} con {pct:.2f}%")


- ¿Cuál es el origen más frecuente entre los buques infractores?
```
El origen más frecuente entre infractores es XXXX con XX registros.
```

In [ ]:
#@title Ejercicio 06.4: Origen más frecuente entre infractores

df = pd.read_csv('port_log/data/interim/port_movements.csv')

origen_counts = df['origen'].value_counts()
origen_top = origen_counts.index[0]
origen_count = origen_counts.iloc[0]

print(f"El origen más frecuente entre infractores es {origen_top} con {origen_count} registros.")



- ¿Cuál es la duración promedio de estadía en muelle de los buques infractores?
```
La duración promedio de estadía de buques infractores es XX.XX horas.
```

In [ ]:
#@title Ejercicio 06.5: Duración promedio de estadía de buques infractores

df = pd.read_csv('port_log/data/interim/port_movements.csv')

# duracion_horas ya está calculada, excluir NaN
duracion_valida = df['duracion_horas'].dropna()
promedio = duracion_valida.mean()

print(f"La duración promedio de estadía de buques infractores es {promedio:.2f} horas.")

## Ejercicio 07

> Puntos: 1

Redactar una conclusión que incluya:

- Evaluación de la calidad del dataset heredado: porcentaje de registros descartados y tipos de error más frecuentes.
- Patrones de infracción detectados: ¿en qué turnos, muelles y tipos de carga se concentran?
- Reflexión sobre el impacto de incorporar estos datos sin limpieza previa al nuevo sistema.
- Al menos una propuesta concreta de mejora para el proceso de captura de datos en el puerto.

Guardar la conclusión en `port_log/reports/conclusion.md`

In [ ]:
#@title Ejercicio 07: Redactar conclusión y guardar en port_log/reports/conclusion.md

conclusion = """# Conclusión - Sprint 1: Port Log Analysis

## 1. Evaluación de la calidad del dataset heredado

El dataset original contenía **1.500 registros** de movimientos portuarios. Tras el proceso de limpieza y normalización, el dataset final quedó con **474 registros válidos**, lo que implica que se descartaron **1.026 registros (68.4%)**.

### Tipos de error más frecuentes:
- **Horas inválidas (18.78% de infracciones)**: valores como `AB:CD`, `30:17:00`, `26:42:00`, formatos 12h/24h mezclados, y nulos (34 en hora_egreso).
- **Fechas inválidas (4.64% de infracciones)**: formatos inconsistentes (`32/13/2021`, `21-05-2020`), días/meses imposibles, y nulos.
- **Matrículas inválidas (91 nulas finales)**: caracteres especiales (`@#$%`), espacios, nulos (56 originales).
- **Valores nulos en columnas críticas**: velocidad_ingreso (82), tonelaje_declarado (69), radar_id (65), estado_despacho (58), matricula (56).
- **Outliers en variables numéricas**: 88 registros eliminados por IQR en tonelaje_declarado y velocidad_ingreso.
- **Registros sin infracción real (772 eliminados)**: exceso_velocidad ≤ 0 tras aplicar tolerancia del 5%.

## 2. Patrones de infracción detectados

### Por turno:
La mayor concentración de infracciones ocurre en **Tarde (27.6%)**, seguida de **Noche (22.4%)**, **Madrugada (22.2%)** y **Mañana (21.7%)**. El turno de tarde muestra mayor volumen, posiblemente por mayor tráfico portuario en ese horario.

### Por muelle:
Los muelles con más infracciones son **MUELLE-D (87, 18.4%)**, **MUELLE-B (82, 17.3%)** y **MUELLE-F (81, 17.1%)**. MUELLE-D destaca con el exceso de velocidad promedio más alto (~3.8).

### Por tipo de carga:
**CONTENEDORES (14.98%)** es el tipo de carga más frecuente en infracciones, seguido de **TRIGO (14.77%)** y **GRANOS (13.08%)**. Las cargas de alto valor/urgencia (contenedores, trigo) tienden a presentar más excesos de velocidad.

## 3. Impacto de incorporar datos sin limpieza

Incorporar estos datos sin limpieza al nuevo sistema tendría consecuencias graves:
- **Cálculos erróneos de duración**: 109 registros con duración NaN por fechas/horas inválidas.
- **Falsos positivos/negativos en infracciones**: horas `AB:CD` o `30:17` generarían errores en clasificación por turno y cálculo de excesos.
- **Pérdida de trazabilidad**: 91 matrículas nulas impiden identificar buques infractores reincidentes.
- **Decisiones operativas sesgadas**: outliers en tonelaje y velocidad distorsionan estadísticas de capacidad y seguridad.
- **Incumplimiento normativo**: reportes oficiales basados en datos sucios podrían derivar en sanciones.

## 4. Propuesta concreta de mejora

**Implementar validación en tiempo real en el sistema de captura portuaria:**
- Controles de formato obligatorios al ingreso: fecha (ISO 8601), hora (24h HH:MM), matrícula (regex alfanumérico+guión).
- Validación cruzada: fecha_egreso ≥ fecha_ingreso, velocidad_ingreso ≤ límite físico del muelle.
- Catálogos normalizados: muelles (MUELLE-A..F), tipos de carga, orígenes predefinidos (dropdown).
- Alertas automáticas: exceso de velocidad > 5% tolerancia notifica a control de tránsito.
- Auditoría diaria: reporte de completitud y calidad de datos capturados en las últimas 24h.
"""

output_path = Path('port_log/reports/conclusion.md')
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    f.write(conclusion)

print(f"Conclusión guardada en: {output_path}")
print("\n=== CONCLUSIÓN ===")
print(conclusion)

---

# CHECKS

`De aquí en adelante solo ejecutar`

## Visualizar la rama actual

In [ ]:
!git status

## Visualizar el contenido de cada commit

In [ ]:
!git --no-pager log --name-status Sprint_1